In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score,mean_absolute_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from scipy.stats import randint,uniform
import joblib
RANDOM_STATE = 42



In [ ]:


def load_data(excel_path: str):
    df = pd.read_excel(excel_path)
    print("Data shape:", df.shape)
    print("Columns:", df.columns.tolist())
    return df


def define_features_targets(df: pd.DataFrame):
    # Adjust these lists if your column names differ
    feature_cols = [f"feat_{i}" for i in range(1, 59)]
    target_cols = [f"width_{i}" for i in range(1, 7)]

    X = df[feature_cols]
    y = df[target_cols]

    return X, y, feature_cols, target_cols


def build_preprocessor(feature_cols):
    numeric_features = feature_cols  # assume all features are numeric

    numeric_transformer = Pipeline(
        steps=[
            ("scaler", StandardScaler())
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features)
        ],
        remainder="drop",
    )

    return preprocessor


def evaluate_model(model, X_train, y_train, X_val, y_val, target_cols, model_name="model"):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)

    print(f"\n===== Evaluation for {model_name} =====")
    print("Overall R^2 (train):", r2_score(y_train, y_train_pred, multioutput='uniform_average'))
    print("Overall R^2 (val):  ", r2_score(y_val, y_val_pred, multioutput='uniform_average'))

    print("\nPer-target metrics:")
    for i, target in enumerate(target_cols):
        r2_trg_train = r2_score(y_train.iloc[:, i], y_train_pred[:, i])
        r2_trg_val = r2_score(y_val.iloc[:, i], y_val_pred[:, i])
        mae_trg_val = mean_absolute_error(y_val.iloc[:, i], y_val_pred[:, i])
        print(
            f"{target:10s} | R2_train={r2_trg_train:.4f} | "
            f"R2_val={r2_trg_val:.4f} | MAE_val={mae_trg_val:.4f}"
        )

    return {
        "y_train_pred": y_train_pred,
        "y_val_pred": y_val_pred,
    }


def main():
    excel_path = "project_file.xlsx"  # change if needed
    df = load_data(excel_path)
    X, y, feature_cols, target_cols = define_features_targets(df)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    preprocessor = build_preprocessor(feature_cols)

    # 1. Baseline: Linear Regression
    baseline_model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("regressor", LinearRegression())
        ]
    )

    evaluate_model(
        baseline_model, X_train, y_train, X_val, y_val, target_cols, model_name="LinearRegression (baseline)"
    )

    # 2. Random Forest (multi-output natively)
    rf = RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    rf_pipeline = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("regressor", rf)
        ]
    )

    rf_param_dist = {
        "regressor__n_estimators": randint(100, 400),
        "regressor__max_depth": randint(3, 15),
        "regressor__min_samples_split": randint(2, 10),
        "regressor__min_samples_leaf": randint(1, 8),
    }

    rf_search = RandomizedSearchCV(
        rf_pipeline,
        param_distributions=rf_param_dist,
        n_iter=20,
        cv=5,
        scoring="r2",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )

    rf_search.fit(X_train, y_train)
    print("\nBest RF params:", rf_search.best_params_)
    best_rf_model = rf_search.best_estimator_

    rf_eval = evaluate_model(
        best_rf_model, X_train, y_train, X_val, y_val, target_cols, model_name="RandomForest (tuned)"
    )

    # 3. Gradient Boosting (wrapped as MultiOutputRegressor)
    gb = GradientBoostingRegressor(random_state=RANDOM_STATE)
    gb_multi = MultiOutputRegressor(gb)

    gb_pipeline = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("regressor", gb_multi)
        ]
    )

    gb_param_dist = {
        "regressor__estimator__n_estimators": randint(100, 400),
        "regressor__estimator__learning_rate": uniform(0.01, 0.2),
        "regressor__estimator__max_depth": randint(2, 5),
    }

    gb_search = RandomizedSearchCV(
        gb_pipeline,
        param_distributions=gb_param_dist,
        n_iter=20,
        cv=5,
        scoring="r2",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )

    gb_search.fit(X_train, y_train)
    print("\nBest GB params:", gb_search.best_params_)
    best_gb_model = gb_search.best_estimator_

    gb_eval = evaluate_model(
        best_gb_model, X_train, y_train, X_val, y_val, target_cols, model_name="GradientBoosting (tuned)"
    )

    # Choose best model by overall validation R^2
    def overall_r2(model, X_val, y_val):
        preds = model.predict(X_val)
        return r2_score(y_val, preds, multioutput='uniform_average')

    rf_r2_val = overall_r2(best_rf_model, X_val, y_val)
    gb_r2_val = overall_r2(best_gb_model, X_val, y_val)

    print("\n===== Model comparison (overall validation R^2) =====")
    print(f"RandomForest:      {rf_r2_val:.4f}")
    print(f"GradientBoosting:  {gb_r2_val:.4f}")

    if rf_r2_val >= gb_r2_val:
        best_model = best_rf_model
        best_name = "RandomForest"
    else:
        best_model = best_gb_model
        best_name = "GradientBoosting"

    print(f"\nSelected best model: {best_name} with R^2 (val) = {max(rf_r2_val, gb_r2_val):.4f}")

    # Save best model
    Path("models").mkdir(exist_ok=True)
    model_path = Path("models") / "best_width_model.pkl"
    joblib.dump(
        {
            "model": best_model,
            "feature_cols": feature_cols,
            "target_cols": target_cols,
        },
        model_path
    )
    print(f"\nSaved best model to: {model_path}")


if __name__ == "__main__":
    main()

Data shape: (100, 64)
Columns: ['feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8', 'feat_9', 'feat_10', 'feat_11', 'feat_12', 'feat_13', 'feat_14', 'feat_15', 'feat_16', 'feat_17', 'feat_18', 'feat_19', 'feat_20', 'feat_21', 'feat_22', 'feat_23', 'feat_24', 'feat_25', 'feat_26', 'feat_27', 'feat_28', 'feat_29', 'feat_30', 'feat_31', 'feat_32', 'feat_33', 'feat_34', 'feat_35', 'feat_36', 'feat_37', 'feat_38', 'feat_39', 'feat_40', 'feat_41', 'feat_42', 'feat_43', 'feat_44', 'feat_45', 'feat_46', 'feat_47', 'feat_48', 'feat_49', 'feat_50', 'feat_51', 'feat_52', 'feat_53', 'feat_54', 'feat_55', 'feat_56', 'feat_57', 'feat_58', 'width_1', 'width_2', 'width_3', 'width_4', 'width_5', 'width_6']

===== Evaluation for LinearRegression (baseline) =====
Overall R^2 (train): 0.7288658908699852
Overall R^2 (val):   -3.4559676831978234

Per-target metrics:
width_1    | R2_train=0.7532 | R2_val=-2.8558 | MAE_val=22.4918
width_2    | R2_train=0.7784 | R2_val=-2.3127 | MAE

In [5]:
# predict_new_records.py

import pandas as pd
import joblib
from pathlib import Path


def load_model(model_path="models/best_width_model.pkl"):
    saved = joblib.load(model_path)
    model = saved["model"]
    feature_cols = saved["feature_cols"]
    target_cols = saved["target_cols"]
    return model, feature_cols, target_cols


def predict_from_excel(excel_path: str, model_path="models/best_width_model.pkl"):
    model, feature_cols, target_cols = load_model(model_path)

    df_new = pd.read_excel(excel_path)

    # Ensure we only pass the trained features
    X_new = df_new[feature_cols]

    preds = model.predict(X_new)

    preds_df = pd.DataFrame(preds, columns=target_cols)
    print("\nPredictions:")
    print(preds_df.head())

    # Optionally save predictions
    output_path = Path("predictions.xlsx")
    preds_df.to_excel(output_path, index=False)
    print(f"\nPredictions saved to {output_path}")


def predict_from_manual_data(model_path="models/best_width_model.pkl"):
    model, feature_cols, target_cols = load_model(model_path)

    # Example: one new record; replace with real values
    sample_data = {
        col: [0.0] for col in feature_cols  # put real feature values here
    }
    df_new = pd.DataFrame(sample_data)

    preds = model.predict(df_new)
    preds_df = pd.DataFrame(preds, columns=target_cols)

    print("\nPredictions for manual sample:")
    print(preds_df)


if __name__ == "__main__":
    # Example usage:
    # 1) Predict from a new Excel file, e.g., new_records.xlsx
    # predict_from_excel("new_records.xlsx")

    # 2) Or from manually defined data
    predict_from_manual_data()



Predictions for manual sample:
     width_1    width_2    width_3    width_4   width_5    width_6
0  26.692988  22.973887  21.067747  23.492236  24.21051  25.018542
